In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [4]:
L0 = 40e3  # mirror length

In [5]:
def radius_of_curvature(coor, value):
    d1 = (value[1:] - value[:-1]) / (coor[1] - coor[0])
    d2 = (d1[1:] - d2[:-1]) / (coor[1] - coor[0])
    R = np.power(1 + d1 ** 2, 1.5) / np.abs(d2)
    return R

def yag_position(D, y0, theta0, theta):
    return (D - L0 / 2. * np.cos(theta0 + theta) + y0 * np.sin(theta0 + theta)) * np.tan(2 * (theta0 + theta))

# M1

In [ ]:
data = np.array([[300, np.deg2rad(0.1)], #(Li  um, thetai  rad)
                 [200, np.deg2rad(0.02)],
                 [400, np.deg2rad(0.2)],
                 [500, np.deg2rad(0.3)],
                 [700, np.deg2rad(0.4)],
                ])

# ----------------------------------
#   Fitting and visualization
def loss(D, y0, theta0):
    angles = theta0 + data[:,1]
    return np.sum(np.square(data[:,0] - (D - L0 / 2. * np.cos(angles) + y0 * np.sin(angles)) * np.tan(2 * (angles))))
    
result = scipy.optimize.minimize(loss,
                                 x0=(4.1e6, 43e3, 0),
                                 method='SLSQP',
                                 bounds=((4e6, 4.4e6), (40e3, 46e3), (-np.deg2rad(1), np.deg2rad(1))),
                                 tol=((4 * 6.5 / 4) ** 2 * float(data.shape[0])),  
                                 # 4 pixel per measurement * 6.5 um pixel size / 5 magnification
                                 # Square * number of point
                                )

# Get the uncertainty of the result by looking at the curve 
angles_scan = np.linspace(-np.deg2rad(0.5), np.deg2rad(0.5), 500) + result[2]
loss_array = loss(result[0], result[1], angles_scan)

radius_of_curve = radius_of_curvature(coor=angles_scan, value=loss_array)
uncertainty = radius_of_curve[angles_scan.shape[0] // 2 + 1]

print("YAG-mirror_center_distance in mm", result[0] / 1000)
print("Rotation center offset in mm", result[1] / 1000)
print("Motor 0deg offset in deg", np.rad2deg(result[2]))


# Visualization
fig, axes = plt.subplots(ncols=3, nrows=1)
fig.set_figheight(4)
fig.set_figwidth(12)

axes[0].plot(angles_scan, loss_array, label='loss', color='b')
axes0_twin = axes[0].twinx()
axes0_twin.plot(angles_scan[1:-1], 2 * radius_of_curvature, label='Uncertainty', color='k')
axes[0].legend()
axes[0].set_title("Loss and uncertainty in theta")

axes[1].plot(data[:,1], data[:,0], label='measure', color='b')
axes[1].plot(data[:,1], yag_position(result[0], result[1], result[2], data[:,1]), label='fit', color='r')
axes[1].set_title("Comparison")
axes[2].legend()

axes[2].plot(data[:,0], yag_position(result[0], result[1], result[2], data[:,1]), color='g')
axes[2].set_title("measure vs fit")
axes[2].legend()

plt.tight_layout()
plt.show()